In [98]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [141]:
import os
import io
import json
import boto3
import glob
from urllib.parse import unquote_plus
from pyspark.sql import SparkSession
import uuid
import sys
import psycopg2
import logging
import pandas as pd
import janitor
from datetime import datetime
from pyspark.sql import SparkSession
from pandas.api.types import infer_dtype
from pyspark.sql.functions import col
import tempfile
from pandas.api.types import infer_dtype
import time
from typing import Optional , Dict, Any, List , Type
import numpy  as np
import awswrangler as wr
import warnings
import hashlib
import pg8000
import unicodedata

In [86]:
os.environ['SECRET_NAME']='rdspiping'
os.environ['AWS_REGION'] = 'eu-north-1'

In [87]:
s3_client = boto3.client('s3')
os.environ['TZ'] = 'UTC'
time.tzset()  # if you import time
ENDPOINT=os.getenv('ENDPOINT')
PORT=int(os.getenv('PORT'))
REGION=os.getenv('REGION')
DBNAME=os.getenv('DBNAME')
DB_PASSWORD=os.getenv('DB_PASSWORD')
SECRET_NAME=os.getenv('SECRET_NAME')
DESTINATION_BUCKETNAME=os.getenv('DESTINATION_BUCKETNAME')

In [13]:
def setup_logger(name: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        formatter = logging.Formatter('[%(asctime)s] %(levelname)s @ line %(lineno)d: %(message)s')
        handler = logging.StreamHandler(sys.stdout)
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

logger = setup_logger(__name__)

In [14]:
def check_environment():
    required_vars = ['ENDPOINT', 'PORT', 'REGION', 'SECRET_NAME', 'DBNAME']
    missing_vars = [var for var in required_vars if not os.getenv(var)]
    if missing_vars:
        logger.error(f"Missing environment variables: {missing_vars}")
        raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")
    else:
        logger.info("All required environment variables are set.")

def get_db_connection_params() -> Dict[str, str]:
    required_params = ['ENDPOINT', 'PORT', 'DBNAME']

    params = {}
    for param in required_params:
        value = os.environ.get(param)
        if not value:
            raise ValueError(f"Missing required environment variable: {param}")
        params[param] = value

    return params

def get_secret(secret_name):

    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=REGION
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        logger.error(f"Failed to retrieve secret: {e}")
        raise
    else:
        if 'SecretString' in get_secret_value_response:
            secret = json.loads(get_secret_value_response['SecretString'])
            return secret
        else:
            raise ValueError("Secret value is not a string")

In [16]:
print("Current directory:", os.getcwd())
print("Files and directories:", os.listdir())

Current directory: /home/jovyan
Files and directories: ['.profile', '.bashrc', '.bash_logout', '.local', '.ipython', 'requirements.txt', '.npm', '.ipynb_checkpoints', '.jupyter', 'RDS.ipynb', 'TEIGA_TMI_ADISSEO_CONTROL_AVANCE_SOPORTACION.csv', '.cache', '.conda', '.config', '.wget-hsts', 'work']


In [125]:
input_path3='TEIGA_TMI_ADISSEO_CONTROL_AVANCE_SOPORTACION.csv'
output_path='output/'

In [49]:
from typing import Union
import pandas as pd
import numpy as np
from datetime import datetime

def infer_column_type(col: pd.Series, threshold: float = 0.95) -> str:
    """
    Infers the data type of a pandas Series column using modern type checking.
    
    Args:
        col: pandas Series to analyze
        threshold: minimum ratio of successful type conversions required
        
    Returns:
        str: Inferred type ('integer', 'float', 'datetime', 'boolean', or 'string')
    """
    # Remove null values for analysis
    clean_col = col.dropna()
    if len(clean_col) == 0:
        return 'string'

    total_rows = len(clean_col)
    
    # Try numeric conversion first
    def check_numeric():
        try:
            # Check if all integers
            pd.to_numeric(clean_col, downcast='integer')
            int_success = np.sum(clean_col.astype(str).str.match(r'^-?\d+$')) / total_rows
            if int_success >= threshold:
                return 'integer'
            
            # Check if floating point
            pd.to_numeric(clean_col, downcast='float')
            float_success = np.sum(clean_col.astype(str).str.match(r'^-?\d+\.?\d*$')) / total_rows
            if float_success >= threshold:
                return 'float'
        except:
            return None
        
    # Check for datetime
    def check_datetime():
        try:
            pd.to_datetime(clean_col)
            datetime_success = np.sum(pd.to_datetime(clean_col, errors='coerce').notna()) / total_rows
            if datetime_success >= threshold:
                return 'datetime'
        except:
            return None

    # Check for boolean
    def check_boolean():
        bool_values = {'true', 'false', '1', '0', 'yes', 'no'}
        bool_success = np.sum(clean_col.astype(str).str.lower().isin(bool_values)) / total_rows
        if bool_success >= threshold:
            return 'boolean'
        return None

    # Try each type in order of priority
    for type_check in [check_numeric, check_datetime, check_boolean]:
        result = type_check()
        if result:
            return result

    # Default to string if no other type matches
    return 'string'

def get_pandas_dtype(type_str: str) -> Union[str, np.dtype]:
    """
    Convert the inferred type string to a pandas-compatible dtype.
    
    Args:
        type_str: The string representation of the type
        
    Returns:
        A pandas-compatible dtype
    """
    dtype_mapping = {
        'integer': 'Int64', 
        'float': 'float64',
        'datetime': 'datetime64[ns]',
        'boolean': 'boolean',
        'string': 'string'
    }
    return dtype_mapping.get(type_str, 'string')


In [71]:
def format_dataframe_columns(df: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    """
    Format and convert DataFrame columns to their appropriate data types.
    
    Args:
        df: Input DataFrame
        threshold: Threshold for type inference (default: 0.95)
        
    Returns:
        DataFrame with properly formatted columns
    """
    formatted_df = df.copy()
    
    # Dictionary to store conversion errors
    conversion_errors = {}
    
    for column in formatted_df.columns:
        try:
            # Infer the type for the column
            inferred_type = infer_column_type(formatted_df[column], threshold)
            pandas_dtype = get_pandas_dtype(inferred_type)
            
            # Convert the column to the inferred type
            if inferred_type == 'datetime':
                formatted_df[column] = pd.to_datetime(formatted_df[column], errors='coerce')
            elif inferred_type == 'boolean':
                formatted_df[column] = formatted_df[column].astype(str).str.lower()
                formatted_df[column] = formatted_df[column].map({
                    'true': True, 'false': False,
                    '1': True, '0': False,
                    'yes': True, 'no': False
                })
            else:
                formatted_df[column] = formatted_df[column].astype(pandas_dtype)
                
            # Log the conversion
            print(f"Column '{column}' converted to {inferred_type} ({pandas_dtype})")
            
        except Exception as e:
            conversion_errors[column] = str(e)
            print(f"Warning: Could not convert column '{column}'. Error: {str(e)}")
            
    if conversion_errors:
        print("\nConversion errors summary:")
        for col, error in conversion_errors.items():
            print(f"Column '{col}': {error}")
    
    return formatted_df

In [83]:
def transformationsETL(df):
    df['RW'] = df.groupby(['E3DID', 'SUPPORTID', 'SUPPORTMARKALL']).cumcount() + 1
    
    df = df[df['E3DID'].notna()]
    
    
    def create_hash(row):
        concat_string = f"{row['E3DID']}||{row['SUPPORTID']}||{row['SUPPORTMARKALL']}"
        return hashlib.sha256(concat_string.encode()).hexdigest()
    
    df['ID_DB'] = df.apply(create_hash, axis=1)
    return df

In [144]:
def clean_column_names(df):
    def clean_name(name):
        # Remove accents
        name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')
        # Replace spaces and special chars with underscore
        name = ''.join(c if c.isalnum() else '_' for c in name)
        # Remove trailing underscores
        name = name.rstrip('_')
        # Ensure name starts with a letter (Athena requirement)
        if name[0].isdigit():
            name = 'col_' + name
        # Remove duplicate underscores
        while '__' in name:
            name = name.replace('__', '_')
        return name.lower()

    df.columns = [clean_name(col) for col in df.columns]
    return df

# Clean the DataFrame column names


In [145]:

def example_usage(df):
    logger.info(f"Original DataFrame dtypes")
    logger.info(f"df columns types:\n{df2.dtypes}")

    # Format the DataFrame
    formatted_df = format_dataframe_columns(df)
    
    print("\nFormatted DataFrame dtypes:")
    print(formatted_df.dtypes)
    print("\nFormatted DataFrame:")
    print(formatted_df)
    
    # Display some basic information about the conversion
    print("\nDataFrame Info:")
    formatted_df.info()

In [146]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
df = pd.read_csv(input_path3, header=0, sep=',')
logger.info(f"Original DataFrame dtypes")
logger.info(f"df columns types:\n{df.dtypes}")
df = transformationsETL(df)
df_format = format_dataframe_columns(df)
df_format = clean_column_names(df_format)
logger.info(f"Formatted DataFrame dtypes")
logger.info(f"formatted_df columns types:\n{df_format.dtypes}")

[2025-04-26 12:16:59,534] INFO @ line 4: Original DataFrame dtypes
[2025-04-26 12:16:59,538] INFO @ line 5: df columns types:
NM_REV                                  object
ID_MONMEGA                              object
TAG                                     object
AREA                                    object
E3DID                                   object
ISOID                                   object
CLASS                                   object
DIAM_ITEM                              float64
SUPPORTID                               object
SUPPORTMARKALL                          object
SUPPORTMARK_or_SIKLATYPE                object
DATA1                                   object
DATA2or_SIKLAPARTNUMBER                 object
DATA3                                   object
DATA4                                   object
DATA5                                   object
DATA6                                   object
DATA7                                   object
DATA8                       

/tmp/ipykernel_6994/2161208316.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ID_DB'] = df.apply(create_hash, axis=1)


Column 'NM_REV' converted to string (string)
Column 'ID_MONMEGA' converted to string (string)
Column 'TAG' converted to string (string)
Column 'AREA' converted to string (string)
Column 'E3DID' converted to string (string)
Column 'ISOID' converted to string (string)
Column 'CLASS' converted to string (string)
Column 'DIAM_ITEM' converted to float (float64)
Column 'SUPPORTID' converted to string (string)
Column 'SUPPORTMARKALL' converted to string (string)
Column 'SUPPORTMARK_or_SIKLATYPE' converted to string (string)
Column 'DATA1' converted to string (string)
Column 'DATA2or_SIKLAPARTNUMBER' converted to string (string)
Column 'DATA3' converted to string (string)
Column 'DATA4' converted to string (string)
Column 'DATA5' converted to string (string)
Column 'DATA6' converted to string (string)
Column 'DATA7' converted to string (string)
Column 'DATA8' converted to string (string)
Column 'DATA9' converted to string (string)
Column 'DATA10' converted to float (float64)
Column 'REMARKS' c

In [147]:
def dropTableIFExist(cur):
    try:
        logger.info("Dropping table user_01.source_support if it exists...")
        cur.execute("DROP TABLE IF EXISTS user_01.source_support")
        logger.info("Table dropped.")
        return True
    except Exception as e:
        logger.error(f"Error dropping table: {str(e)}")
        return False
    
def createTableSupport(cur, df_format, conn):
    try:
        logger.info("Creating new source_support table...")

        # Map pandas dtypes to SQL types
        dtype_mapping = {
            'int64': 'INTEGER',
            'float64': 'FLOAT',
            'object': 'VARCHAR(100)',
            'bool': 'BOOLEAN',
            'datetime64[ns]': 'TIMESTAMP'
        }

        columns_def = ', '.join(
            f"{col} {dtype_mapping.get(str(dtype), 'TEXT')}"
            for col, dtype in df_format.dtypes.items()
        )

        create_table_query = f"""
            CREATE TABLE user_01.source_support (
                id SERIAL PRIMARY KEY,
                {columns_def}
            )
        """
        cur.execute(create_table_query)
        return True
    except Exception as e:
        logger.error(f"Error creating table: {str(e)}")
        return False

def loadDataSupport(cur, df_format, conn):
    try:
        logger.info("Loading data into source_support...")
    
        buffer = io.StringIO()
        df_format.to_csv(buffer, index=False, header=True)
        buffer.seek(0) 
    
        columns = ', '.join(df_format.columns)
        copy_sql = f"""
        COPY user_01.source_support({columns})
        FROM STDIN WITH CSV HEADER
        """
        cur.copy_expert(copy_sql, buffer)
        logger.info("Data successfully loaded into database.")
        return True
    except Exception as e:
        logger.error(f"Error loading data support to table user_01.source_support: {str(e)}")
        return False

In [148]:
credentials = get_secret(os.environ.get("SECRET_NAME"))
def pusblishSupportTable(df_format):
    
    # conn = pg8000.connect(database=DBNAME, host=ENDPOINT, port=PORT, user=credentials['username'], password=credentials['password'])
    # cur = conn.cursor()

    db_params = get_db_connection_params()
    conn = psycopg2.connect(
        host=db_params['ENDPOINT'],
        port=db_params['PORT'],
        database=db_params['DBNAME'],
        user=credentials['username'],
        password=credentials['password'],
        sslmode='require'
    )
    cur = conn.cursor()


    
    if not dropTableIFExist(cur):
        logger.error("Failed to drop the table. Aborting publish.")
        cur.close()
        conn.close()
    if not createTableSupport(cur,df_format,conn):
        logger.error("Failed to create the table. Aborting publish.")
        cur.close()
        conn.close()
    if not loadDataSupport(cur, df_format, conn):
        logger.error("Failed to load data to the table. Aborting publish.")
        cur.close()
        conn.close()
    
    
    # for cmd in sqlCmds:
    #     print("Executing Query:" + cmd)
    #     cursor.execute(cmd)
    #     print("Completed Executing Query:" + cmd)
        
    #queryOuts = cursor.fetchall()
    cur.close()
    conn.commit()
        
    #for out in queryOuts:
    #    print(out)


In [149]:
pusblishSupportTable(df_format)

[2025-04-26 12:17:12,113] INFO @ line 3: Dropping table user_01.source_support if it exists...
[2025-04-26 12:17:12,342] INFO @ line 5: Table dropped.
[2025-04-26 12:17:12,348] INFO @ line 13: Creating new source_support table...
[2025-04-26 12:17:12,488] INFO @ line 43: Loading data into source_support...
[2025-04-26 12:17:13,951] INFO @ line 55: Data successfully loaded into database.


In [131]:
database_name = "piping_db_glue"
description = "This is my Glue database for Piping DB"

# Simple creation
wr.catalog.create_database(
    name=database_name,
    description=description
)


In [137]:
glue_database = "piping_db_glue"
glue_table = "iceberg_support"
bucket_name= "control-piping-2025"
path = f"s3://{bucket_name}/iceberg_test/"
temp_path = f"s3://{bucket_name}/iceberg_test_temp/"


In [142]:


# Print cleaned column names to verify
print("Cleaned column names:", df_format.columns.tolist())


Cleaned column names: ['nm_rev', 'id_monmega', 'tag', 'area', 'e3did', 'isoid', 'class', 'diam_item', 'supportid', 'supportmarkall', 'supportmark_or_siklatype', 'data1', 'data2or_siklapartnumber', 'data3', 'data4', 'data5', 'data6', 'data7', 'data8', 'data9', 'data10', 'remarks', 'vigasparacartelas', 'old_support_id', 'fabricante', 'estado_de_fabricacion_nuevo_formato', 'no_entrega_nuevo_formato', 'cuenta', 'fecha', 'montaje', 'fecha2', 'peso_kg', 'observacion', 'conteo', 'weld', 'iv', 'fecha_iv', 'result', 'pt', 'fecha_pt', 'result_pt', 'test_pack_asociado', 'support', 'rw', 'id_db']


In [143]:
glue_database = "piping_db_glue"
glue_table = "iceberg_support"
bucket_name = "control-piping-2025"
path = f"s3://{bucket_name}/iceberg_test/"
temp_path = f"s3://{bucket_name}/iceberg_test_temp/"

try:
    wr.athena.to_iceberg(
        df=df_format,
        database=glue_database,
        table=glue_table,
        table_location=path,
        temp_path=temp_path,
        mode='overwrite',
        workgroup='primary',
        schema_evolution=True
    )
except Exception as e:
    print(f"Error: {str(e)}")

In [ ]:
wr.athena.to_iceberg

In [ ]:
wr.catalog.sc

In [120]:

from pyspark.sql.functions import col, when, isnan, lit,isnull, trim, regexp_replace

def load_to_database_spark(df_spark):
    try:

        df_sample = df_spark.sample(withReplacement=False, fraction=0.3, seed=42).toPandas()

        # Step 2: Infer column types with janitor
        inferred_types = {}
        for col_name in df_sample.columns:
            dtype = infer_dtype(df_sample[col_name], skipna=True)
            logger.info(f"Inferred type for column {col_name}: {dtype}")
            inferred_types[col_name] = dtype

        # Step 3: Optional - cast types in Spark DataFrame based on inferred types
        cast_mapping = {
            'string': 'string',
            'unicode': 'string',
            'mixed': 'string',
            'mixed-integer': 'string',
            'integer': 'int',
            'floating': 'double',
            'boolean': 'boolean',
            'datetime64': 'TIMESTAMP_NTZ'
        }

        for col_name, inferred in inferred_types.items():
            spark_type = cast_mapping.get(inferred, 'string')
            df_spark = df_spark.withColumn(col_name, col(col_name).cast(spark_type))

        logger.info(f"Columns after casting: {df_spark.dtypes}")

        # Step 4: Save Spark DataFrame as a single CSV file
        with tempfile.TemporaryDirectory() as tmp_dir:
            output_path = os.path.join(tmp_dir, "spark_output")


            for c, dtype in df_spark.dtypes:
                if dtype == 'string':
                    df_spark = df_spark.withColumn(c, trim(col(c)))
                    df_spark = df_spark.withColumn(c, regexp_replace(col(c), r'["\n\r]', ' '))
            df_spark.coalesce(1).write.csv(output_path, header=True, mode='overwrite')
            # Find the CSV part file
            part_file = next((f for f in os.listdir(output_path) if f.startswith("part-") and f.endswith(".csv")), None)
            if not part_file:
                raise Exception("CSV part file not found")

            csv_file_path = os.path.join(output_path, part_file)

            # Step 5: Connect to database
            conn = psycopg2.connect(
                host=ENDPOINT,
                port=PORT,
                database=DBNAME,
                user=USER,
                password='Asimo1721.',
                sslmode='require'
            )
            cur = conn.cursor()

            # Step 6: Create table if it doesn't exist
            check_table_query = """
            SELECT EXISTS (
                SELECT FROM information_schema.tables 
                WHERE table_schema= 'public'
                AND table_name = 'source_support'
            );
            """
            cur.execute(check_table_query)
            table_exist = cur.fetchone()[0]

            if not table_exist:
                create_table_query = """
                    CREATE TABLE public.source_support (
                        id SERIAL PRIMARY KEY,
                        {}
                    )
                """.format(
                    ', '.join([f"{col} TEXT" for col in df_spark.columns])
                )
                cur.execute(create_table_query)
                conn.commit()

            # Step 7: Load data via COPY
            with open(csv_file_path, 'r', encoding='utf-8') as f:
                logger.info("Using COPY to load data into source_support...")
                cur.copy_expert(f"""
                    COPY public.source_support({', '.join(df_spark.columns)})
                    FROM STDIN WITH CSV HEADER
                """, f)

            conn.commit()
            logger.info("Data successfully loaded with COPY")

    except Exception as e:
        logger.error(f"Database operation failed: {e}")
        raise

    finally:
        if 'cur' in locals() and cur:
            cur.close()
        if 'conn' in locals() and conn:
            conn.close()


In [12]:
import tempfile
import os
from pyspark.sql.functions import col, when, isnan, lit, isnull, trim, regexp_replace
import janitor
from pandas.api.types import infer_dtype
import psycopg2
from typing import Dict, Optional
import logging

# Configure logging
logger = logging.getLogger(__name__)

def get_db_connection_params() -> Dict[str, str]:
    """Get database connection parameters from environment variables."""
    required_params = ['ENDPOINT', 'PORT', 'DBNAME', 'USER', 'DB_PASSWORD']
    
    params = {}
    for param in required_params:
        value = os.environ.get(param)
        if not value:
            raise ValueError(f"Missing required environment variable: {param}")
        params[param] = value
    
    return params

def load_to_database_spark(df_spark):
    conn = None
    cur = None
    
    try:
        # Step 1: Sample and infer types
        df_sample = df_spark.sample(withReplacement=False, fraction=0.3, seed=42).toPandas()
        
        # Step 2: Infer column types
        inferred_types = {
            col_name: infer_dtype(df_sample[col_name], skipna=True)
            for col_name in df_sample.columns
        }
        
        for col_name, dtype in inferred_types.items():
            logger.info(f"Inferred type for column {col_name}: {dtype}")

        # Step 3: Cast types in Spark DataFrame
        cast_mapping = {
            'string': 'string',
            'unicode': 'string',
            'mixed': 'string',
            'mixed-integer': 'string',
            'integer': 'int',
            'floating': 'double',
            'boolean': 'boolean',
            'datetime64': 'TIMESTAMP_NTZ'
        }
        logger.info(f"Row count before write: {df_spark.count()}")

        for col_name, inferred in inferred_types.items():
            spark_type = cast_mapping.get(inferred, 'string')
            df_spark = df_spark.withColumn(col_name, col(col_name).cast(spark_type))

        logger.info(f"Columns after casting: {df_spark.dtypes}")

        # Step 4: Clean and save data
        with tempfile.TemporaryDirectory() as tmp_dir:
            output_path = os.path.join(tmp_dir, "spark_output")

            # Clean string columns
            for c, dtype in df_spark.dtypes:
                if dtype == 'string':
                    df_spark = df_spark.withColumn(
                        c, 
                        regexp_replace(trim(col(c)), r'["\n\r]', ' ')
                    )

            df_spark.coalesce(1).write.csv(output_path, header=True, mode='overwrite')
            
            # Find CSV part file
            part_file = next(
                (f for f in os.listdir(output_path) 
                 if f.startswith("part-") and f.endswith(".csv")),
                None
            )
            
            if not part_file:
                raise FileNotFoundError("CSV part file not found")

            csv_file_path = os.path.join(output_path, part_file)

            # Step 5: Connect to database using environment variables
            db_params = get_db_connection_params()
            conn = psycopg2.connect(
                host=db_params['ENDPOINT'],
                port=db_params['PORT'],
                database=db_params['DBNAME'],
                user=db_params['USER'],
                password=db_params['DB_PASSWORD'],
                sslmode='require'
            )
            cur = conn.cursor()

            # Step 6: Create table if doesn't exist
            cur.execute("""
                SELECT EXISTS (
                    SELECT FROM information_schema.tables 
                    WHERE table_schema = 'public'
                    AND table_name = 'source_support'
                );
            """)
            
            if not cur.fetchone()[0]:
                columns_def = ', '.join(f"{col} TEXT" for col in df_spark.columns)
                create_table_query = f"""
                    CREATE TABLE public.source_support (
                        id SERIAL PRIMARY KEY,
                        {columns_def}
                    )
                """
                cur.execute(create_table_query)
                conn.commit()

            # Step 7: Load data via COPY
            with open(csv_file_path, 'r', encoding='utf-8') as f:
                logger.info("Loading data into source_support...")
                columns = ', '.join(df_spark.columns)
                copy_sql = f"""
                    COPY public.source_support({columns})
                    FROM STDIN WITH CSV HEADER
                """
                cur.copy_expert(copy_sql, f)

            conn.commit()
            logger.info("Data successfully loaded")

    except Exception as e:
        if conn:
            conn.rollback()
        logger.error(f"Database operation failed: {str(e)}")
        raise

    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()


In [32]:
# from pyspark.sql.functions import col
# from pandas.api.types import infer_dtype

# def load_to_database_spark(df_spark):
#     try:
#         # Clean column names
#         column_mapping = {col_name: clean_column_name(col_name) for col_name in df_spark.columns}
#         df_spark = df_spark.toDF(*[column_mapping[c] for c in df_spark.columns])

#         # Optional: show inferred types per column (Spark -> Pandas for display)
#         df_pandas_preview = df_spark.limit(100).toPandas()
#         for col_name in df_pandas_preview.columns:
#             logger.info(f"{col_name}: {infer_dtype(df_pandas_preview[col_name])}")

#         logger.info(f"CSV columns cleaned: {df_spark.columns}")

#         # Convert full Spark DataFrame to Pandas for DB load (if data size is small)
#         df_clean = df_spark.toPandas()

#         # Handle nulls
#         df_clean = df_clean.where(pd.notnull(df_clean), None)

#         # Connect to DB
#         conn = psycopg2.connect(
#             host=ENDPOINT,
#             port=PORT,
#             database=DBNAME,
#             user=USER,
#             password='Asimo1721.',
#             sslmode='require'
#         )
#         cur = conn.cursor()

#         # Check if table exists
#         check_table_query = """
#         SELECT EXISTS (
#             SELECT FROM information_schema.tables 
#             WHERE table_schema= 'public'
#             AND table_name = 'source_support'
#         );
#         """
#         cur.execute(check_table_query)
#         table_exist = cur.fetchone()[0]

#         if not table_exist:
#             create_table_query = """
#                 CREATE TABLE public.source_support (
#                     id SERIAL PRIMARY KEY,
#                     {}
#                 )
#             """.format(
#                 ', '.join([f"{col} TEXT" for col in df_clean.columns])
#             )
#             cur.execute(create_table_query)
#             conn.commit()

#         logger.info("Inserting data into source_support")
#         columns = ', '.join(df_clean.columns)
#         values = ','.join(['%s' for _ in df_clean.columns])
#         insert_query = f"""
#             INSERT INTO public.source_support ({columns})
#             VALUES ({values})
#         """

#         records = df_clean.values.tolist()
#         cur.executemany(insert_query, records)
#         conn.commit()

#         logger.info("Data successfully loaded to database")

#     except Exception as e:
#         logger.error(f"Database operation failed: {e}")
#         raise
#     finally:
#         if 'cur' in locals() and cur:
#             cur.close()
#         if 'conn' in locals() and conn:
#             conn.close()


In [20]:
def process_csv(input_path, output_path):

    try:


        df_spark = sparkjum.read.csv(input_path, header=True, inferSchema=True)
        df_spark = df_spark.na.drop(how="all")
        logger.info(f"Row count before FUCNTION load_to_database_spark: {df_spark.count()}")


        logger.info(f"CSV columns: {df_spark.columns}")
        load_to_database_spark(df_spark)
        # df = df.groupby('E3DID').agg({'MONTAJE': 'sum'})
        # df.to_csv(output_path, index=False)
      
        

        return True
    except Exception as e:
        logger.error(f"Error processing CSV: {str(e)}")
        return False

In [19]:
process_csv(input_path,output_path)

[2025-04-23 11:47:38,306] INFO @ line 8: Row count before FUCNTION load_to_database_spark: 9461
[2025-04-23 11:47:38,310] INFO @ line 11: CSV columns: ['NM_REV', 'ID_MONMEGA', 'TAG', 'AREA', 'E3DID', 'ISOID', 'CLASS', 'DIAM_ITEM', 'SUPPORTID', 'SUPPORTMARKALL', 'SUPPORTMARK_or_SIKLATYPE', 'DATA1', 'DATA2or_SIKLAPARTNUMBER', 'DATA3', 'DATA4', 'DATA5', 'DATA6', 'DATA7', 'DATA8', 'DATA9', 'DATA10', 'REMARKS', 'VIGASparaCARTELAS', 'OLD_SUPPORT_ID', 'FABRICANTE', 'ESTADO_DE_FABRICACION_NUEVO_FORMATO', 'No_ENTREGA_NUEVO_FORMATO', 'Cuenta', 'FECHA', 'MONTAJE', 'FECHA2', 'PESO_KG', 'Observación', 'CONTEO', 'Weld', 'IV', 'Fecha_IV', 'Result', 'PT_', 'Fecha_PT', 'Result_PT', 'TEST_PACK_ASOCIADO_', 'SUPPORT']


/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):


[2025-04-23 11:47:38,868] INFO @ line 41: Inferred type for column NM_REV: string
[2025-04-23 11:47:38,869] INFO @ line 41: Inferred type for column ID_MONMEGA: string
[2025-04-23 11:47:38,870] INFO @ line 41: Inferred type for column TAG: string
[2025-04-23 11:47:38,872] INFO @ line 41: Inferred type for column AREA: string
[2025-04-23 11:47:38,874] INFO @ line 41: Inferred type for column E3DID: string
[2025-04-23 11:47:38,875] INFO @ line 41: Inferred type for column ISOID: string
[2025-04-23 11:47:38,876] INFO @ line 41: Inferred type for column CLASS: string
[2025-04-23 11:47:38,878] INFO @ line 41: Inferred type for column DIAM_ITEM: string
[2025-04-23 11:47:38,878] INFO @ line 41: Inferred type for column SUPPORTID: string
[2025-04-23 11:47:38,879] INFO @ line 41: Inferred type for column SUPPORTMARKALL: string
[2025-04-23 11:47:38,880] INFO @ line 41: Inferred type for column SUPPORTMARK_or_SIKLATYPE: string
[2025-04-23 11:47:38,881] INFO @ line 41: Inferred type for column DAT

True